# 01 · Thu thập dữ liệu

- **Người phụ trách:** Đỗ Khánh Duy
- **Đầu vào:** API công khai của Chợ Tốt (`gateway.chotot.com/v1/public/ad-listing`)
- **Đầu ra:** `data/raw/cars.csv` (15.000 tin × 35 trường); `data/raw/descriptions.csv` (mô tả dài theo `list_id`, dùng ở notebook 04)
- **Liên quan proposal:** mục 3 (Dữ liệu) và phương án xử lý rủi ro "không có trường mô tả dài" ở mục 5

In [1]:
import json
from pathlib import Path

import pandas as pd

# Tìm gốc repo (thư mục có requirements.txt) để đường dẫn đúng trên máy mọi thành viên
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "requirements.txt").exists())

RAW_PATH = ROOT / "data" / "raw" / "cars.csv"
DESCRIPTIONS_PATH = ROOT / "data" / "raw" / "descriptions.csv"
STATUS_PATH = ROOT / "data" / "raw" / "description_status.csv"
REPORT_PATH = ROOT / "data" / "raw" / "descriptions_report.json"

raw = pd.read_csv(RAW_PATH, low_memory=False)
print(f"Dữ liệu gốc: {len(raw):,} dòng × {len(raw.columns)} cột")


Dữ liệu gốc: 15,000 dòng × 35 cột


## 2. Cào bổ sung mô tả dài

Bộ dữ liệu gốc có `subject` nhưng chưa có mô tả dài. Theo phương án xử lý rủi ro ở mục 5 của proposal, `crawler/fetch_descriptions.py` đọc các `list_id` có sẵn trong `data/raw/cars.csv`, truy vấn chi tiết từng tin và lưu nguyên văn mô tả lấy được. Crawler không thay mô tả thiếu bằng tiêu đề.

Các đầu ra phục vụ kiểm tra và chạy tiếp:

- `data/raw/descriptions.csv`: đúng hai cột `list_id,description`, chỉ chứa tin có mô tả không rỗng và có trạng thái `ok`.
- `data/raw/description_status.csv`: một dòng cho mỗi mã tin gốc, gồm `list_id,status,http_status,attempts,checked_at,source_url,error`. File này ghi cả các tin chưa kiểm tra (`pending`).
- `data/raw/descriptions_report.json`: thông tin lần chạy, số lượng theo trạng thái và lý do dừng.

**Cách chạy đầy đủ.** Từ gốc repository, chạy lệnh dưới đây trong terminal để kiểm tra toàn bộ 15.000 mã tin. Lần thu thập này sử dụng tối đa 4 yêu cầu đồng thời, với khoảng cách tối thiểu 0,25 giây giữa hai lần bắt đầu yêu cầu trên toàn bộ crawler. Khi gặp HTTP 429, các worker cùng tạm chờ theo `Retry-After` (tối thiểu 60 giây). Các cell của notebook chỉ đọc kết quả trên đĩa, nên **Run All không tự gửi yêu cầu mạng**.

```bash
python 01_crawler_and_ml/crawler/fetch_descriptions.py --workers 4 --delay 0.25 --timeout 20 --max-attempts 3
```

Lệnh không đặt giới hạn thời gian hoặc số tin. Nếu cần một lượt ngắn để kiểm tra, có thể thêm `--max-seconds 600`; đây là tùy chọn, không phải phạm vi của lượt thu thập đầy đủ. Crawler lưu checkpoint sau từng kết quả và xuất CSV định kỳ; khi bị ngắt hoặc gặp 10 kết quả lỗi liên tiếp, lý do dừng được ghi trong báo cáo. Chạy lại cùng lệnh sẽ tiếp tục từ tiến độ đã lưu: bỏ qua `ok`, `removed`, `empty`, `unavailable` và thử lại các trạng thái lỗi. Thêm `--refresh` khi muốn kiểm tra lại cả những kết quả đã kết thúc. Trạng thái phản ánh thời điểm `checked_at`, có thể khác thời điểm thu thập `cars.csv`.

**Quy ước thống kê.** Chỉ HTTP 404/410 được ghi là `removed` (không còn truy cập được chi tiết theo mã tin); đây là tiêu chí quan sát của đợt cào, không xác định ai gỡ tin hoặc nguyên nhân. `unavailable` là chi tiết trả về trạng thái không hoạt động, chưa đủ căn cứ kết luận đã gỡ. Tin còn hiển thị nhưng không có nội dung mô tả được ghi `empty`. Các lỗi HTTP khác, lỗi mạng, giới hạn tốc độ và phản hồi không hợp lệ không được cộng vào số tin đã gỡ.

Bảng dưới được tính lại từ các file bàn giao, không ước lượng số tin đã gỡ từ phần dữ liệu đã kiểm tra.

In [2]:
# Chỉ đọc file cục bộ; chạy crawler trong terminal nếu chưa có đầu ra.
required_paths = [DESCRIPTIONS_PATH, STATUS_PATH, REPORT_PATH]
missing_paths = [str(p.relative_to(ROOT)) for p in required_paths if not p.exists()]
if missing_paths:
    raise FileNotFoundError("Chưa có đầu ra crawler: " + ", ".join(missing_paths))

descriptions = pd.read_csv(DESCRIPTIONS_PATH, dtype={"list_id": "string"}, keep_default_na=False)
status = pd.read_csv(STATUS_PATH, dtype="string", keep_default_na=False)
report = json.loads(REPORT_PATH.read_text(encoding="utf-8"))

status_order = [
    "ok", "removed", "empty", "unavailable", "http_error", "network_error",
    "invalid_response", "rate_limited", "pending",
]
status_labels = {
    "ok": "Có mô tả dài",
    "removed": "Chi tiết không còn truy cập được (HTTP 404/410)",
    "empty": "Tin hiển thị nhưng mô tả rỗng",
    "unavailable": "Tin không hoạt động, chưa rõ nguyên nhân",
    "http_error": "Lỗi HTTP khác",
    "network_error": "Lỗi mạng hoặc hết thời gian chờ",
    "invalid_response": "Phản hồi không hợp lệ",
    "rate_limited": "Bị giới hạn tốc độ",
    "pending": "Chưa kiểm tra",
}
expected_status_columns = {
    "list_id", "status", "http_status", "attempts", "checked_at", "source_url", "error",
}
assert list(descriptions.columns) == ["list_id", "description"], "Sai cấu trúc descriptions.csv"
assert expected_status_columns == set(status.columns), "Sai cấu trúc nhật ký trạng thái"
assert raw["list_id"].notna().all() and raw["list_id"].is_unique, "Mã tin gốc thiếu hoặc trùng"
assert status["list_id"].is_unique and descriptions["list_id"].is_unique, "Mã tin đầu ra bị trùng"
raw_ids = set(raw["list_id"].astype("string"))
assert set(status["list_id"]) == raw_ids, "Nhật ký phải phủ đúng mọi mã tin gốc"
assert set(status["status"]) <= set(status_order), "Có trạng thái chưa được định nghĩa"
assert descriptions["description"].astype("string").str.strip().ne("").all(), "Mô tả thành công bị rỗng"
ok_ids = set(status.loc[status["status"].eq("ok"), "list_id"])
assert set(descriptions["list_id"]) == ok_ids, "Mô tả không khớp các mã tin có trạng thái ok"
removed_rows = status.loc[status["status"].eq("removed")]
assert removed_rows["http_status"].isin(["404", "410"]).all(), "Không đủ bằng chứng HTTP để tính tin đã gỡ"

counts = status["status"].value_counts().reindex(status_order, fill_value=0).astype(int)
total = len(raw_ids)
checked = total - int(counts["pending"])
error_states = ["http_error", "network_error", "invalid_response", "rate_limited"]
error_count = int(counts.loc[error_states].sum())
coverage = 100 * len(descriptions) / total if total else 0.0
summary = pd.DataFrame({
    "Trạng thái": status_order,
    "Ý nghĩa": [status_labels[s] for s in status_order],
    "Số tin": counts.to_numpy(),
    "% toàn bộ tin": (counts.to_numpy() / total * 100).round(2) if total else 0.0,
})
assert int(summary["Số tin"].sum()) == total

print(f"Dữ liệu gốc: {len(raw):,} dòng × {len(raw.columns)} cột; {total:,} mã tin duy nhất.")
print(f"Đã có kết quả kiểm tra: {checked:,}/{total:,} tin; chưa kiểm tra: {int(counts['pending']):,} tin.")
print(f"Có mô tả dài: {len(descriptions):,}/{total:,} tin ({coverage:.2f}%).")
print(f"Tin đã gỡ theo tiêu chí HTTP 404/410: {int(counts['removed']):,} tin.")
print(summary.to_string(index=False))
checked_times = status.loc[status["checked_at"].ne(""), "checked_at"]
if not checked_times.empty:
    print(f"Khoảng thời gian ghi nhận kết quả: {checked_times.min()} đến {checked_times.max()}.")
print("\nThông tin lần chạy được lưu trong descriptions_report.json:")
print(json.dumps(report, ensure_ascii=False, indent=2))

print("\nGiới hạn và phạm vi sử dụng:")
if int(counts["pending"]) or error_count:
    print(
        f"Chưa có kết quả chắc chắn cho {int(counts['pending']) + error_count:,} tin "
        f"({int(counts['pending']):,} chưa kiểm tra; {error_count:,} gặp lỗi). "
        "Không suy rộng số tin đã gỡ hoặc tỷ lệ có mô tả của phần đã kiểm tra ra toàn bộ dữ liệu."
    )
else:
    print("Mọi mã tin đã có kết quả kiểm tra; các kết quả chỉ phản ánh thời điểm cào bổ sung.")
print(
    f"Có {int(counts['unavailable']):,} tin không hoạt động chưa rõ nguyên nhân và "
    f"{int(counts['empty']):,} tin có mô tả rỗng; không gộp hai nhóm này vào tin đã gỡ."
)
print(
    "Notebook 04 nối trái theo list_id, báo độ phủ trên tập dữ liệu thực tế dùng cho NLP; "
    "các tin thiếu mô tả có thể phân tích bằng subject và phải nêu rõ giới hạn. "
    "Mô tả thu được muộn hơn dữ liệu gốc có thể đã thay đổi, và nhóm còn truy cập được "
    "có thể không đại diện cho mọi tin trong tập gốc."
)

Dữ liệu gốc: 15,000 dòng × 35 cột; 15,000 mã tin duy nhất.
Đã có kết quả kiểm tra: 15,000/15,000 tin; chưa kiểm tra: 0 tin.
Có mô tả dài: 10,547/15,000 tin (70.31%).
Tin đã gỡ theo tiêu chí HTTP 404/410: 4,453 tin.
      Trạng thái                                         Ý nghĩa  Số tin  % toàn bộ tin
              ok                                    Có mô tả dài   10547          70.31
         removed Chi tiết không còn truy cập được (HTTP 404/410)    4453          29.69
           empty                   Tin hiển thị nhưng mô tả rỗng       0           0.00
     unavailable        Tin không hoạt động, chưa rõ nguyên nhân       0           0.00
      http_error                                   Lỗi HTTP khác       0           0.00
   network_error                 Lỗi mạng hoặc hết thời gian chờ       0           0.00
invalid_response                           Phản hồi không hợp lệ       0           0.00
    rate_limited                              Bị giới hạn tốc độ       0         

## 2. Kết luận

`data/raw/cars.csv` bàn giao dữ liệu tin đăng gốc cho notebook 02. `data/raw/descriptions.csv` bổ sung nội dung dài cho notebook 04 qua khóa `list_id`; số mô tả, độ phủ và số tin đã gỡ quan sát được nằm trong kết quả thống kê ở mục 4. `description_status.csv` và `descriptions_report.json` giữ bằng chứng về phạm vi kiểm tra, lỗi chưa xử lý và lý do dừng, đồng thời giúp tiếp tục thu thập.

Phạm vi phân tích văn bản phụ thuộc vào số mô tả thực sự thu được. Notebook 04 cần giữ các tin thiếu mô tả khi nối dữ liệu, phân biệt mô tả dài với tiêu đề và công bố tỷ lệ bao phủ trên tập dùng cho Word Cloud hoặc biến cờ. Không có mô tả không đồng nghĩa tin đã bị gỡ; các trường hợp chưa kiểm tra hoặc lỗi truy cập vẫn là giới hạn của lần bàn giao này.